# 7. Evaluation & Conclusion
## CRISP-DM Phases 5 & 6
This notebook synthesizes the results from all previous phases — EDA, Data Preparation, PCA, K-Means, and Decision Tree — into a cohesive evaluation with actionable business insights and a final conclusion.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)

data_dir = 'data'
print("Libraries loaded.")


## 7.1 Project Recap

| CRISP-DM Phase | Notebook | Key Output |
|---|---|---|
| 2. Data Understanding | `new_eda.ipynb` | Relational dataset explored; FM aggregation validated |
| 3. Data Preparation | `data_preparation.ipynb` | One-Hot encoded, StandardScaled feature matrix |
| 4a. PCA | `pca.ipynb` | Reduced from N features → 7 principal components (≥90% variance) |
| 4b. K-Means | `kmeans.ipynb` | Customer segments identified via Elbow + Silhouette |
| 4c. Decision Tree | `new_decision_tree.ipynb` | Interpretable rules explaining cluster membership |


## 7.2 Cluster Quality Evaluation
We evaluate the quality of our K-Means segmentation using multiple metrics to ensure the clusters are meaningful and not arbitrary.


In [ ]:
# Load datasets
clusters_df = pd.read_csv(os.path.join(data_dir, 'customer_clusters.csv'))
pca_df = pd.read_csv(os.path.join(data_dir, 'customer_pca_features.csv'))
if 'CustomerID' in pca_df.columns:
    pca_df = pca_df.set_index('CustomerID')

X_pca = pca_df.values
labels = clusters_df['Cluster'].values

# Silhouette Score
sil_score = silhouette_score(X_pca, labels)
print(f"Overall Silhouette Score: {sil_score:.4f}")
print(f"  → Score > 0.25 = reasonable structure")
print(f"  → Score > 0.50 = strong structure")
print(f"  → Score > 0.70 = excellent structure")

# Cluster sizes
print(f"\nCluster Sizes:")
print(clusters_df['Cluster'].value_counts().sort_index())
print(f"\nCluster Balance (std of sizes): {clusters_df['Cluster'].value_counts().std():.1f}")


## 7.3 Customer Segment Profiles
The following analysis provides a comprehensive view of each customer segment, combining numerical behavioral metrics with categorical preferences.


In [ ]:
# Numerical profile per cluster
num_profile = clusters_df.groupby('Cluster').agg(
    Size=('CustomerID', 'count'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Expenditure=('Total_Expenditure', 'mean'),
    Avg_AOV=('Average_Order_Value', 'mean'),
    Avg_Items=('Total_Items_Bought', 'mean')
).round(2)

print("=" * 70)
print("NUMERICAL SEGMENT PROFILES")
print("=" * 70)
print(num_profile.to_string())


In [ ]:
# Categorical profile per cluster
print("\n" + "=" * 70)
print("CATEGORICAL SEGMENT PROFILES")
print("=" * 70)

for cluster_id in sorted(clusters_df['Cluster'].unique()):
    subset = clusters_df[clusters_df['Cluster'] == cluster_id]
    top_region = subset['Region'].value_counts().head(2)
    top_category = subset['Preferred_Category'].value_counts().head(2)
    print(f"\n--- Cluster {cluster_id} ({len(subset)} customers) ---")
    print(f"  Top Regions:    {dict(top_region)}")
    print(f"  Top Categories: {dict(top_category)}")


## 7.4 Visual Summary of Segments


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Avg Expenditure per Cluster
num_profile['Avg_Expenditure'].plot(kind='bar', ax=axes[0, 0], color='teal', edgecolor='black')
axes[0, 0].set_title('Average Total Expenditure per Segment')
axes[0, 0].set_ylabel('Avg Expenditure ($)')
axes[0, 0].tick_params(axis='x', rotation=0)

# 2. Avg Frequency per Cluster
num_profile['Avg_Frequency'].plot(kind='bar', ax=axes[0, 1], color='coral', edgecolor='black')
axes[0, 1].set_title('Average Purchase Frequency per Segment')
axes[0, 1].set_ylabel('Avg Frequency')
axes[0, 1].tick_params(axis='x', rotation=0)

# 3. Avg AOV per Cluster
num_profile['Avg_AOV'].plot(kind='bar', ax=axes[1, 0], color='mediumpurple', edgecolor='black')
axes[1, 0].set_title('Average Order Value per Segment')
axes[1, 0].set_ylabel('Avg AOV ($)')
axes[1, 0].tick_params(axis='x', rotation=0)

# 4. Cluster sizes
num_profile['Size'].plot(kind='bar', ax=axes[1, 1], color='goldenrod', edgecolor='black')
axes[1, 1].set_title('Customer Count per Segment')
axes[1, 1].set_ylabel('Number of Customers')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.suptitle('Customer Segment Comparison Dashboard', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()


### 7.4.1 Segment Radar Chart
A radar (spider) chart gives a normalized multi-dimensional view of how each segment compares across all behavioral metrics simultaneously.


In [ ]:
from matplotlib.patches import FancyBboxPatch

# Normalize metrics to 0-1 for radar chart
metrics = ['Avg_Frequency', 'Avg_Expenditure', 'Avg_AOV', 'Avg_Items']
radar_df = num_profile[metrics].copy()
for col in metrics:
    min_val = radar_df[col].min()
    max_val = radar_df[col].max()
    if max_val > min_val:
        radar_df[col] = (radar_df[col] - min_val) / (max_val - min_val)
    else:
        radar_df[col] = 0.5

# Radar plot
categories = ['Frequency', 'Expenditure', 'AOV', 'Items Bought']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors = plt.cm.tab10(np.linspace(0, 1, len(radar_df)))

for idx, (cluster_id, row) in enumerate(radar_df.iterrows()):
    values = row.values.tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=f'Cluster {cluster_id}', color=colors[idx])
    ax.fill(angles, values, alpha=0.1, color=colors[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=11)
ax.set_title('Normalized Segment Radar Chart', size=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()


## 7.5 Decision Tree Model Evaluation

The Decision Tree classifier was trained to predict cluster membership using the original (interpretable) customer features. Key evaluation points:

1. **Accuracy:** The classification accuracy on the held-out test set indicates how well-defined and separable the K-Means clusters are. High accuracy means the clusters follow clear, rule-based behavioral patterns.

2. **Feature Importance:** The most important features in the Decision Tree reveal the **primary drivers** of customer segmentation — these are the metrics that marketing teams should focus on.

3. **Interpretable Rules:** Each path from root to leaf in the Decision Tree describes a customer archetype in plain business language (e.g., "If a customer's total expenditure exceeds $X and they buy more than Y items, they belong to the high-value segment").


## 7.6 Business Recommendations

Based on the complete analysis pipeline, we recommend the following strategic actions:

### Segment-Specific Strategies
- **High-Value Segments:** Implement loyalty programs and exclusive offers to retain these customers. They drive disproportionate revenue.
- **Low-Frequency Segments:** Design re-engagement campaigns (email, promotions) to increase purchase frequency.
- **Category-Focused Segments:** Personalize product recommendations based on each segment's preferred category.
- **Regional Segments:** Tailor marketing language, currency, and shipping options to the dominant region of each segment.

### Data-Driven Decision Making
- Use the Decision Tree rules as a **scoring model** to classify new customers into segments automatically.
- Monitor segment migration over time — customers moving from low-value to high-value segments indicates successful marketing.
- The PCA components can serve as a **dimensionality-reduced input** for future advanced models (e.g., neural networks, gradient boosting).


## 7.7 Conclusion

This project successfully demonstrated a complete **CRISP-DM data mining workflow** for customer segmentation in e-commerce:

1. **Data Understanding (EDA):** We explored a relational dataset of 200 customers, 100 products, and 1,000 transactions across 4 regions and 5 product categories. Key distributions and behavioral patterns were identified.

2. **Data Preparation:** Raw transactional data was aggregated into customer-level behavioral profiles (FM Analysis), categorically encoded, and standardized using `StandardScaler` — ensuring mathematical fairness across all features.

3. **PCA (Dimensionality Reduction):** Principal Component Analysis compressed the feature space while preserving ≥90% of the dataset's variance, eliminating noise and multi-collinearity.

4. **K-Means (Customer Segmentation):** Unsupervised clustering, validated by both the Elbow Method and Silhouette Analysis, identified distinct customer segments with measurably different behavioral profiles.

5. **Decision Tree (Behavioral Explanation):** A supervised classifier translated the abstract cluster assignments into human-readable business rules, revealing which features drive segmentation and enabling automated classification of new customers.

### Limitations
- The dataset is relatively small (200 customers), which limits the statistical power of the segmentation.
- Time-based factors (seasonality, recency) were excluded per project constraints, which would typically enrich behavioral analysis.
- The Decision Tree's depth was capped for interpretability, potentially sacrificing some classification accuracy.

### Future Work
- Incorporate temporal features (RFM analysis) when constraints allow.
- Test alternative clustering algorithms (DBSCAN, Hierarchical) for comparison.
- Deploy the Decision Tree as a real-time customer scoring API.
- Expand the dataset with more customers and transaction history for more robust segments.
